# SVM: расширенная широкая сетка и плотное уточнение


In [ ]:
import os
import time
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from joblib import Memory, dump, parallel_backend
from threadpoolctl import threadpool_limits
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    GridSearchCV, ParameterGrid, StratifiedKFold,
    RepeatedStratifiedKFold, cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.svm import SVC

SEED = 20260916
SEARCH_JOBS = min(2, os.cpu_count() or 1)
print("scikit-learn:", sklearn.__version__)
print("Параллельных обучений:", SEARCH_JOBS)

In [ ]:
DATA_DIR = Path(".")

train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

feature_columns = [c for c in train.columns if c not in {"row_id", "target"}]
X_train = train[feature_columns]
y_train = train["target"]
X_validation = validation[feature_columns]
y_validation = validation["target"]
X_test = test[feature_columns]

assert X_train.shape == (2520, 512)
assert X_validation.shape == (560, 512)
assert X_test.shape == (1116, 512)
assert not train.isna().any().any()
assert not validation.isna().any().any()
assert not test.isna().any().any()
assert set(y_train.unique()) == {0, 1}
assert set(y_validation.unique()) == {0, 1}

cv_splits = list(
    StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    .split(X_train, y_train)
)
print("train:", X_train.shape, "CV folds:", len(cv_splits))

for frame in (X_train, X_validation, X_test):
    assert np.isfinite(frame.to_numpy()).all()

RUN_DIR = DATA_DIR / "runs" / time.strftime("%Y%m%d-%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=False)

CACHE_DIR = Path(mkdtemp(prefix="svm_"))
memory = Memory(CACHE_DIR, verbose=0)
print("Результаты этого запуска:", RUN_DIR.resolve())

## Широкая сетка


In [ ]:
def make_pipeline(scale="passthrough", reduce="passthrough", kernel="rbf"):
    return Pipeline([
        ("scale", scale),
        ("reduce", reduce),
        ("svc", SVC(
            kernel=kernel, degree=2, cache_size=512, random_state=SEED,
        )),
    ], memory=memory)

c_values = [0.01, 0.1, 0.3, 1.0, 2.5, 5.0, 10.0, 30.0, 100.0]
gamma_values = [0.0001, 0.0003, 0.001, 0.003, 0.005, 0.007, 0.01, 0.03, 0.1]
rbf_grid = {"svc__C": c_values, "svc__gamma": gamma_values}

branches = {
    "rbf_raw": (make_pipeline(), rbf_grid),
    "rbf_standard": (make_pipeline(StandardScaler()), rbf_grid),
    "rbf_robust": (make_pipeline(RobustScaler()), rbf_grid),
    "rbf_pca128": (
        make_pipeline(StandardScaler(), PCA(
            n_components=128, svd_solver="randomized", random_state=SEED
        )), rbf_grid,
    ),
    "rbf_pca256_whiten": (
        make_pipeline(StandardScaler(), PCA(
            n_components=256, whiten=True,
            svd_solver="randomized", random_state=SEED
        )), rbf_grid,
    ),
}
poly_grid = {
    "svc__C": [0.1, 1.0, 10.0],
    "svc__gamma": [0.001, 0.007, 0.03],
    "svc__coef0": [0.0, 1.0],
}
branches["poly2_raw"] = (make_pipeline(kernel="poly"), poly_grid)
branches["poly2_standard"] = (
    make_pipeline(StandardScaler(), kernel="poly"), poly_grid
)

display(pd.DataFrame([
    {"Вариант": name, "Комбинаций": len(ParameterGrid(grid)),
     "Обучений CV": len(ParameterGrid(grid)) * len(cv_splits)}
    for name, (_, grid) in branches.items()
]))

def run_grid(name, estimator, grid, stage):
    count = len(ParameterGrid(grid))
    print(f"\n{stage} / {name}: {count} комбинаций, "
          f"{count * len(cv_splits)} обучений CV", flush=True)
    search = GridSearchCV(
        clone(estimator), grid, scoring="roc_auc", cv=cv_splits,
        n_jobs=SEARCH_JOBS, pre_dispatch=SEARCH_JOBS,
        refit=True, error_score="raise", verbose=0,
        return_train_score=False,
    )
    started = time.perf_counter()

    with threadpool_limits(limits=1), parallel_backend("threading"):
        search.fit(X_train, y_train)
    search.search_seconds_ = time.perf_counter() - started
    pd.DataFrame(search.cv_results_).sort_values(
        "rank_test_score"
    ).to_csv(RUN_DIR / f"{stage}_{name}_results.csv", index=False)
    dump(search, RUN_DIR / f"{stage}_{name}_search.joblib")
    print(f"{name}: CV={search.best_score_:.6f}, "
          f"время={search.search_seconds_ / 60:.1f} мин", flush=True)
    return search

def search_table(searches):
    return pd.DataFrame([
        {"Вариант": name, "CV ROC-AUC": search.best_score_,
         "CV std": search.cv_results_["std_test_score"][search.best_index_],
         "Время, мин": search.search_seconds_ / 60,
         "Параметры": search.best_params_}
        for name, search in searches.items()
    ]).sort_values("CV ROC-AUC", ascending=False).reset_index(drop=True)

In [ ]:
coarse_searches = {}
for name, (estimator, grid) in branches.items():
    coarse_searches[name] = run_grid(name, estimator, grid, "coarse")
    search_table(coarse_searches).to_csv(
        RUN_DIR / "coarse_summary.csv", index=False
    )
display(search_table(coarse_searches))

## Узкая сетка вокруг трёх лучших вариантов

Для каждого из трёх лучших вариантов широкого поиска проверяем C от 0.4× до 2.5× найденного значения и gamma от 0.5× до 2× найденного значения.
В каждой оси 9 точек, включая исходный центр: 81 комбинация на вариант. Преобразование, ядро и coef0 фиксируются на найденных значениях.

Широкий победитель остаётся кандидатом. Узкая сетка выполняется один раз; она не расширяется в цикле.

In [ ]:
assert len(coarse_searches) == len(branches), "Сначала завершите широкий поиск."
top_names = search_table(coarse_searches)["Вариант"].head(3).tolist()
fine_searches = {}
for name in top_names:
    coarse = coarse_searches[name]
    center = coarse.best_params_
    grid = {
        "svc__C": np.unique(
            np.r_[center["svc__C"] * np.geomspace(0.4, 2.5, 9),
                  center["svc__C"]].round(12)
        ).tolist(),
        "svc__gamma": np.unique(
            np.r_[center["svc__gamma"] * np.geomspace(0.5, 2.0, 9),
                  center["svc__gamma"]].round(12)
        ).tolist(),
    }
    assert len(ParameterGrid(grid)) == 81
    fine_searches[name] = run_grid(
        name, coarse.best_estimator_, grid, "fine"
    )
    search_table(fine_searches).to_csv(RUN_DIR / "fine_summary.csv", index=False)
display(search_table(fine_searches))

## Повторная CV для финалистов


In [ ]:
assert set(fine_searches) == set(top_names), "Сначала завершите узкий поиск."
candidates = {
    "previous_rbf": make_pipeline().set_params(svc__C=2.5, svc__gamma=0.007)
}
for name in top_names:
    winner = max(
        [coarse_searches[name], fine_searches[name]],
        key=lambda search: search.best_score_,
    )
    candidates[name] = clone(winner.best_estimator_)

confirm_splits = list(RepeatedStratifiedKFold(
    n_splits=5, n_repeats=2, random_state=SEED + 101
).split(X_train, y_train))

confirmation_scores = {}
for name, estimator in candidates.items():
    print(f"Повторная CV: {name}, {len(confirm_splits)} обучений", flush=True)
    with threadpool_limits(limits=1), parallel_backend("threading"):
        confirmation_scores[name] = cross_val_score(
            estimator, X_train, y_train, scoring="roc_auc",
            cv=confirm_splits, n_jobs=SEARCH_JOBS,
            pre_dispatch=SEARCH_JOBS, error_score="raise", verbose=0,
        )
    pd.DataFrame(confirmation_scores).to_csv(
        RUN_DIR / "confirmation_fold_scores.csv", index=False
    )
    print(f"Средний ROC-AUC: {confirmation_scores[name].mean():.6f}", flush=True)

baseline_scores = confirmation_scores["previous_rbf"]
confirmation = pd.DataFrame([
    {"Вариант": name, "Repeated CV ROC-AUC": scores.mean(),
     "CV std": scores.std(ddof=1),
     "Средняя разница с baseline": (scores - baseline_scores).mean(),
     "Фолдов лучше baseline": int((scores > baseline_scores).sum())}
    for name, scores in confirmation_scores.items()
]).sort_values("Repeated CV ROC-AUC", ascending=False, kind="stable")
confirmation.to_csv(RUN_DIR / "confirmation_summary.csv", index=False)
display(confirmation)

selected_name = max(
    confirmation_scores, key=lambda name: confirmation_scores[name].mean()
)
selected_svm = clone(candidates[selected_name])
print("Выбрана модель:", selected_name)
print("Параметры SVC:", {key: selected_svm.named_steps["svc"].get_params()[key]
      for key in ("kernel", "C", "gamma", "degree", "coef0")})

## Финальная проверка и вероятности


In [ ]:
calibrated_svm = CalibratedClassifierCV(
    estimator=clone(selected_svm), method="sigmoid",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    ensemble=False, n_jobs=SEARCH_JOBS,
)
started = time.perf_counter()
with threadpool_limits(limits=1), parallel_backend("threading"):
    calibrated_svm.fit(X_train, y_train)
    previous_svm = clone(candidates["previous_rbf"]).fit(X_train, y_train)

selected_validation_probability = calibrated_svm.predict_proba(X_validation)[:, 1]
selected_validation_auc = roc_auc_score(y_validation, selected_validation_probability)
previous_validation_auc = roc_auc_score(
    y_validation, previous_svm.decision_function(X_validation)
)
comparison = pd.DataFrame([
    {"Модель": "Прежний RBF: C=2.5, gamma=0.007",
     "Repeated CV ROC-AUC": baseline_scores.mean(),
     "Validation ROC-AUC": previous_validation_auc},
    {"Модель": selected_name,
     "Repeated CV ROC-AUC": confirmation_scores[selected_name].mean(),
     "Validation ROC-AUC": selected_validation_auc},
])
display(comparison)
comparison.to_csv(RUN_DIR / "final_comparison.csv", index=False)
dump(calibrated_svm, RUN_DIR / "selected_calibrated_svm.joblib")
print(f"Обучение и калибровка: {time.perf_counter() - started:.1f} с")
print(f"Validation ROC-AUC: {selected_validation_auc:.6f}")
print(f"Разница с прежней моделью: {selected_validation_auc - previous_validation_auc:+.6f}")

assert selected_validation_probability.shape == (len(validation),)
assert np.isfinite(selected_validation_probability).all()
assert ((selected_validation_probability >= 0) & (selected_validation_probability <= 1)).all()

In [ ]:
test_probability = calibrated_svm.predict_proba(X_test)[:, 1]

submission_svm_dense = sample_submission.copy()
submission_svm_dense["target"] = test_probability

assert list(submission_svm_dense.columns) == ["row_id", "target"]
assert submission_svm_dense.shape == (len(test), 2)
assert submission_svm_dense["row_id"].equals(test["row_id"])
assert submission_svm_dense["row_id"].is_unique
assert not submission_svm_dense.isna().any().any()
assert np.isfinite(test_probability).all()
assert submission_svm_dense["target"].between(0, 1).all()

submission_name = f"submission_svm_{selected_name}_auc_{selected_validation_auc:.6f}_{RUN_DIR.name}.csv"
submission_svm_dense_path = RUN_DIR / submission_name
submission_svm_dense.to_csv(submission_svm_dense_path, index=False)
print("Submission для Kaggle (AUC в имени — локальная validation):")
print(submission_svm_dense_path.resolve())
display(submission_svm_dense.head())